In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [ ]:
# @title Генерация синусоиды

import numpy as np
import pandas as pd
from ipywidgets import interact, IntSlider

import numpy as np
import pandas as pd

def generate_sinusoid_with_jump(
    length: int = 1000,
    base_period: float = 50,
    freq: str = 'H',
    start_time: str = '2024-01-01 00:00:00',
    noise_level: float = 0.1,
    freq_mod_period: int = 300,
    freq_mod_strength: float = 0.3,
    amp_mod_period: int = 500,
    amp_mod_strength: float = 0.5,
    add_trend: bool = True,
    trend_segments: int = 3,
    add_seasonality: bool = True,
    season_period: int = 144,
    season_amplitude: float = 0.5,
    random_seed: int = None
) -> pd.DataFrame:
    if random_seed is not None:
        np.random.seed(random_seed)

    index = pd.date_range(start=start_time, periods=length, freq=freq)
    t = np.arange(length)

    # --- Частотная модуляция ---
    freq_mod = 1 + freq_mod_strength * np.sin(2 * np.pi * t / freq_mod_period)
    phase = 2 * np.pi * np.cumsum(freq_mod) / base_period

    # --- Амплитудная модуляция ---
    amp_mod = 1 + amp_mod_strength * np.sin(2 * np.pi * t / amp_mod_period)

    # --- Основной сигнал ---
    signal = amp_mod * np.sin(phase)

    # --- Тренд с переломами ---
    if add_trend:
        trend = np.zeros(length)
        segment_len = length // trend_segments
        current_value = 0.0
        for i in range(trend_segments):
            slope = np.random.uniform(-0.03, 0.03)
            start = i * segment_len
            end = length if i == trend_segments - 1 else (i + 1) * segment_len
            seg_t = np.arange(end - start)
            seg_values = current_value + slope * seg_t
            trend[start:end] = seg_values
            current_value = seg_values[-1]  # продолжаем без разрывов
        signal += trend

    # --- Сезонность ---
    if add_seasonality:
        seasonal = season_amplitude * np.sin(2 * np.pi * t / season_period)
        signal += seasonal

    # --- Шум ---
    signal += noise_level * np.random.randn(length)

    return pd.DataFrame({"signal": signal}, index=index)



df = generate_sinusoid_with_jump(
    length=10000,
    freq='H',
    base_period=50,
    freq_mod_strength=0.2,
    amp_mod_strength=0.2,
    trend_segments=20,
    season_period=168,   # недельная сезонность (при freq='H')
    season_amplitude=0.7,
    noise_level=0.15,
    random_seed=42
)

def plot_data(start_idx=0, end_idx=100):
    subset = df.iloc[start_idx:end_idx]
    # Визуализация
    plt.figure(figsize=(12, 4))
    plt.plot(subset["signal"], label="Generated Signal")
    plt.title("Sinusoid with Jump")
    plt.xlabel("Time Step")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.show()

interact(
    plot_data,
    start_idx=IntSlider(min=0, max=len(df)-1, step=1, value=0, description='Start'),
    end_idx=IntSlider(min=1, max=len(df), step=1, value=100, description='End')
);

/tmp/ipython-input-2-2424326261.py:30: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  index = pd.date_range(start=start_time, periods=length, freq=freq)


interactive(children=(IntSlider(value=0, description='Start', max=9999), IntSlider(value=100, description='End…

In [ ]:
# @title Функции создания фичей

def add_rolling_mean(df: pd.DataFrame, column: str, window: int) -> pd.DataFrame:
    df[f'{column}_ma_{window}'] = df[column].rolling(window=window).mean()
    return df

def add_rolling_min(df: pd.DataFrame, column: str, window: int) -> pd.DataFrame:
    df[f'{column}_min_{window}'] = df[column].rolling(window=window).min()
    return df

def add_rolling_max(df: pd.DataFrame, column: str, window: int) -> pd.DataFrame:
    df[f'{column}_max_{window}'] = df[column].rolling(window=window).max()
    return df

def add_rolling_std(df: pd.DataFrame, column: str, window: int) -> pd.DataFrame:
    df[f'{column}_std_{window}'] = df[column].rolling(window=window).std()
    return df

def add_velocity(df: pd.DataFrame, signal_col: str, time_col: str) -> pd.DataFrame:
    df[f'{signal_col}_velocity'] = np.gradient(df[signal_col], df[time_col])
    return df

def efficiency_ratio(series: pd.Series, window: int) -> pd.Series:
    """
    Расчёт коэффициента Efficiency Ratio (ER) по Kaufman.

    Аргументы:
        series (pd.Series): исходный временной ряд (например, цена или сигнал)
        window (int): размер окна

    Возвращает:
        pd.Series: коэффициент ER (0...1), NaN в начале
    """
    # Направленное изменение
    direction = series.diff(window).abs()

    # Шум: сумма абсолютных изменений внутри окна
    volatility = series.diff().abs().rolling(window=window).sum()

    # ER = направленное / суммарное
    er = direction / volatility

    return er

def compute_absolute_speed(signal: pd.Series):
    """
    Абсолютная скорость (разность значений) при равномерной шкале.
    v_t = y_t - y_{t-1}
    """
    speed = np.empty_like(signal.values)
    speed[:] = np.nan
    speed[1:] = signal.values[1:] - signal.values[:-1]
    return speed


def compute_absolute_acceleration(signal: pd.Series):
    """
    Абсолютное ускорение (вторая разность) при равномерной шкале.
    a_t = y_t - 2*y_{t-1} + y_{t-2}
    """
    acceleration = np.empty_like(signal.values)
    acceleration[:] = np.nan
    acceleration[2:] = signal.values[2:] - 2 * signal.values[1:-1] + signal.values[:-2]
    return acceleration

def compute_relative_speed(signal: pd.Series):
    """
    Относительная скорость (доходность).
    v_t = (y_t / y_{t-1}) - 1
    """
    speed = np.empty_like(signal.values)
    speed[:] = np.nan
    speed[1:] = (signal.values[1:] / signal.values[:-1]) - 1
    return speed


def compute_relative_acceleration(signal: pd.Series):
    """
    Относительное ускорение (разность относительных скоростей).
    a_t = v_t - v_{t-1}, где v_t = доходность
    """
    rel_speed = compute_relative_speed(signal)
    acceleration = np.empty_like(signal.values)
    acceleration[:] = np.nan
    acceleration[2:] = rel_speed[2:] - rel_speed[1:-1]
    return acceleration

def calculate_rsi(df: pd.DataFrame, column_name, window=14):
    # Вычисляем разницу между текущим и предыдущим значением цены
    delta = df[column_name].diff()

    # Выбираем положительные и отрицательные изменения
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    # Скользящее среднее приростов и падений
    avg_gain = gain.rolling(window=window, min_periods=window).mean()
    avg_loss = loss.rolling(window=window, min_periods=window).mean()

    # Вычисляем RS и RSI
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    # Добавляем RSI как новую колонку в DataFrame
    new_col_name = f"{column_name}_{window}"
    df[new_col_name] = rsi

    return df

def calculate_bollinger_bands(df: pd.DataFrame, column_name, window=20, num_std=2):
    # Среднее значение за окно (SMA)
    middle_band = df[column_name].rolling(window=window).mean()

    # Стандартное отклонение за окно
    std_dev = df[column_name].rolling(window=window).std()

    # Расчет полос
    upper_band = middle_band + std_dev * num_std
    lower_band = middle_band - std_dev * num_std

    # Названия новых колонок
    prefix = f"{column_name}_bollinger_"
    col_upper = f"upper_{window}_{num_std}".replace('.', 'p')
    col_middle = f"middle_{window}_{num_std}".replace('.', 'p')
    col_lower = f"lower_{window}_{num_std}".replace('.', 'p')

    # Добавляем в DataFrame
    df[f"{prefix}{col_upper}"] = upper_band
    df[f"{prefix}{col_middle}"] = middle_band
    df[f"{prefix}{col_lower}"] = lower_band

    return df

def causal_loess(y, window=20, degree=1):
    y = np.asarray(y)
    result = np.full_like(y, np.nan, dtype=np.float64)
    for t in range(window, len(y)):
        X = np.arange(t - window , t).reshape(-1, 1)
        Y = y[t - window : t]

        model = LinearRegression()
        model.fit(X, Y)

        result[t] = model.predict([[t]])[0]  # Прогноз только на текущий момент
    return result


def compute_efficiency_ratio(df: pd.DataFrame, window: int, column_name:str = 'Close') -> pd.Series:
    # Направленное изменение
    direction = df[column_name].diff(window).abs()

    # Шум: сумма абсолютных изменений внутри окна
    volatility = df[column_name].diff().abs().rolling(window=window).sum()

    # ER = направленное / суммарное
    er = direction / volatility

    er.fillna(0.00001, inplace=True)
    er.replace(0, 0.00001, inplace=True)

    return er


def compute_log_bolinger(df: pd.DataFrame, window: int, num_std=2, k=0.1, column_name: str = 'Close'):
    rolling_std = df[column_name].rolling(window=window).std()
    rolling_mean = df[column_name].rolling(window=window).mean()
    upper_band = rolling_mean + num_std * rolling_std
    lower_band = rolling_mean - num_std * rolling_std
    bb_width = np.log(upper_band / lower_band) * k

    bb_width.fillna(0.00001, inplace=True)
    bb_width.replace(0, 0.00001, inplace=True)

    return bb_width


In [ ]:
PATH = "/content/drive/MyDrive/stocks/Data/DOGEUSDT/preprocessed/DOGEUSDT_1m_2024-01-01_to_2025-05-23_agg_15min_bb_20_2_shifted.joblib"
df = joblib.load(PATH)

In [ ]:

# df['timestamp'] = (df.index.astype(int) / 1e9).astype(int)

for window in [20] :
  df = add_rolling_mean(df, column='Close', window=window)
  # df = add_rolling_min(df, column='Close', window=window)
  # df = add_rolling_max(df, column='Close', window=window)
  df = add_rolling_std(df, column='Close', window=window)

  df = compute_absolute_speed(df['Close'])
  df = compute_absolute_acceleration(df['Close'])

  # df = calculate_rsi(df, 'Close', window=14)
  # df = calculate_bollinger_bands(df, 'Close', window=14, num_std=2)
  df[f'casual_loess_{window}'] =  causal_loess(df['Close'], window = window)
  df[f'residual_{window}'] = df['signal'] - df[f'casual_loess_{window}']


# Таргет как среднее со сдвигом влево
# df['target_avg'] = df['Close_ma_14'].shift(-7)
# df['target_std'] = df['Close_ma_std'].shift(-3)



In [ ]:
df[f'high_trend'] =  causal_loess(df['signal'], window = 240)
df[f'low_trend'] =  causal_loess(df['signal'], window = 15)

df[f'high_trend_noise'] = df['signal'] - df[f'high_trend']
df[f'low_trend_noise'] = df['signal'] - df[f'low_trend']

df.dropna(inplace=True)


In [ ]:
df = add_rolling_mean(df, column='high_trend_noise', window=60)
df = add_rolling_min(df, column='high_trend_noise', window=60)
df = add_rolling_max(df, column='high_trend_noise', window=60)
df = add_rolling_std(df, column='high_trend_noise', window=60)

df = add_rolling_mean(df, column='low_trend', window=5)
df = add_rolling_min(df, column='low_trend', window=5)
df = add_rolling_max(df, column='low_trend', window=5)
df = add_rolling_std(df, column='low_trend', window=5)




In [ ]:
df['signal_abs_speed'] = compute_absolute_speed(df['signal'])
df['signal_abs_accel'] = compute_absolute_acceleration(df['signal'])
df['signal_rel_speed'] = compute_relative_speed(df['signal'])
df['signal_rel_accel'] = compute_relative_acceleration(df['signal'])





In [ ]:
from ipywidgets import interact, IntSlider
# Интерактивный график через ipywidgets
# ----------------------------

features = titles = ['signal', 'signal_abs_speed', 'signal_abs_accel', 'signal_rel_speed', 'signal_rel_accel' ]#df.columns


def plot_data(start_idx=0, end_idx=100):
    subset = df.iloc[start_idx:end_idx]

    fig, axes = plt.subplots(len(features), 1, figsize=(15, len(features)*3), sharex=True)
    for ax, feature, title in zip(axes, features, titles):
        ax.plot(subset['timestamp'], subset[feature], label=title)
        ax.set_title(title)
        ax.grid(True)
    plt.tight_layout()
    plt.show()

interact(
    plot_data,
    start_idx=IntSlider(min=0, max=len(df)-1, step=1, value=0, description='Start'),
    end_idx=IntSlider(min=1, max=len(df), step=1, value=100, description='End')
);

interactive(children=(IntSlider(value=0, description='Start', max=9759), IntSlider(value=100, description='End…

In [ ]:
df.columns

Index(['signal', 'timestamp', 'high_trend', 'low_trend', 'high_trend_noise',
       'low_trend_noise', 'high_trend_noise_ma_60', 'high_trend_noise_min_60',
       'high_trend_noise_max_60', 'high_trend_noise_std_60', 'low_trend_ma_5',
       'low_trend_min_5', 'low_trend_max_5', 'low_trend_std_5', 'signal_speed',
       'signal_abs_accel', 'signal_rel_speed', 'signal_rel_accel',
       'signal_abs_speed'],
      dtype='object')

In [ ]:
joblib.dump(df, '/content/drive/MyDrive/stocks/duet_regression/data/SINUS_04.joblib')

['/content/drive/MyDrive/stocks/duet_regression/data/SINUS_04.joblib']

In [ ]:
# восстановление сигнала из скорости

import pandas as pd
from typing import Any, Tuple
import matplotlib.pyplot as plt


def plot_signal_prediction_windows(
    model,
    data: pd.DataFrame,
    config: Any,
    signal_col: str = 'signal',
    n: int = 5,
    device: str = "cuda"
):
    """
    Выбирает n случайных окон из data, делает предикт скоростей и ускорений,
    рекурсивно восстанавливает сигнал и рисует графики:
    - исходный сигнал (входное окно)
    - истинный сигнал (на горизонте)
    - предсказанный сигнал (на горизонте)

    Ожидается, что:
    - config имеет атрибуты: seq_len, horizon, signal_col
    - predict.predict_window(model, x_window, config, device) возвращает (speed, acceleration)
    """

    seq_len = config.seq_len
    signal_col = signal_col
    horizon = config.horizon

    max_start = len(data) - seq_len - horizon
    indices = np.random.choice(max_start, size=n, replace=False)

    for idx_example, i in enumerate(indices):
        window_df = data.iloc[i:i + seq_len + horizon]
        x_window_df = window_df.iloc[:seq_len]  # вход
        y_true_df = window_df.iloc[seq_len:seq_len + horizon]  # правильный таргет

        # Получаем предикт скоростей и ускорений

        pred = predict.predict_window(model, x_window_df, config, device=device)
        print(pred.shape)
        signal_abs_speed, signal_abs_accel = np.split(pred, 2, axis=1)

        # Рекурсивно восстанавливаем сигнал
        p0 = x_window_df[signal_col].iloc[-1]
        pred_signal = np.zeros(len(signal_abs_speed))
        pred_signal[0] = p0 + signal_abs_speed[0].item()

        for t in range(1, len(signal_abs_speed)):
            pred_signal[t] = (
                pred_signal[t - 1]
                + signal_abs_speed[t].item()
                + 0.5 * signal_abs_accel[t].item()
            )

        # Временные оси
        t_input = np.arange(seq_len)
        t_forecast = np.arange(seq_len, seq_len + horizon)

        # График
        plt.figure(figsize=(10, 4))
        plt.plot(t_input, x_window_df[signal_col], label="Input Signal", color='blue')
        plt.plot(t_forecast, y_true_df[signal_col], label="True Signal", color='green', marker='o')
        plt.plot(t_forecast, pred_signal, label="Predicted Signal", color='red', linestyle='--', marker='x')
        plt.title(f"Example {idx_example + 1}: {signal_col.upper()} Forecast")
        plt.xlabel("Time Step")
        plt.ylabel("Signal Value")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

plot_signal_prediction_windows(
    model=model,
    data=df,
    config=config,
    signal_col = 'efficiency_ratio_14',
    n=30,
    device="cuda"
)